# Pipeline Runner

Interactive notebook for testing and running the data pipelines locally.
Useful for debugging individual steps or running a full pipeline without deploying.

In [ ]:
import sys
import os
import logging

# Add pipelines root to path so shared modules are importable
pipelines_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if pipelines_root not in sys.path:
    sys.path.insert(0, pipelines_root)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

import pandas as pd
pd.set_option("display.max_rows", 50, "display.max_columns", None)

from shared.season import current_season_year, is_in_season
from shared.constants import DEFAULT_SPANS, META_LABELS, STAT_LABELS

print(f"Season: {current_season_year()}")
print(f"In season: {is_in_season()}")
print(f"Default spans: {DEFAULT_SPANS}")

## Configuration

Set the season, teams, and spans you want to work with.

In [ ]:
SEASON = current_season_year()
IS_WOMENS = False
SPANS = DEFAULT_SPANS  # [3, 5, 7]

# Load team keys from CSV
gender = "womens" if IS_WOMENS else "mens"
teams_csv = os.path.abspath(os.path.join(pipelines_root, "..", "data", f"{gender}_teams.csv"))
teams_df = pd.read_csv(teams_csv)
ALL_TEAMS = teams_df["SR key"].tolist()

# Pick a subset for testing (set to ALL_TEAMS for full run)
TEST_TEAMS = ALL_TEAMS[:3]

print(f"Season: {SEASON}")
print(f"Gender: {'Women' if IS_WOMENS else 'Men'}")
print(f"Total teams: {len(ALL_TEAMS)}")
print(f"Test teams: {TEST_TEAMS}")
teams_df.head()

---
## Team Stats Pipeline

Step-by-step execution of the team stats pipeline.
Each cell runs one stage so you can inspect intermediate results.

### Step 1 — Download gamelogs

Scrapes basic + advanced gamelog HTML from Sports Reference.
⚠️ This makes HTTP requests with a 3s delay per team — use `TEST_TEAMS` for quick iteration.

In [ ]:
from shared.scraper import download_gamelogs

teams_to_run = TEST_TEAMS  # Change to ALL_TEAMS for full run

download_gamelogs(teams_to_run, SEASON, IS_WOMENS)
print(f"Downloaded gamelogs for {len(teams_to_run)} teams")

### Step 2 — Parse basic gamelogs

Converts downloaded HTML into clean CSVs.

In [ ]:
from shared.parser import create_basic_gamelog
from shared.scraper import get_team_season_file_path

for key in teams_to_run:
    try:
        create_basic_gamelog(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except (ValueError, FileNotFoundError) as e:
        print(f"✗ {key}: {e}")

# Preview one
sample_key = teams_to_run[0]
csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_basic.csv", IS_WOMENS)
pd.read_csv(csv_path).head()

### Step 3 — Parse advanced gamelogs

In [ ]:
from shared.parser import create_advanced_gamelog

for key in teams_to_run:
    try:
        create_advanced_gamelog(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except (ValueError, FileNotFoundError) as e:
        print(f"✗ {key}: {e}")

csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_advanced.csv", IS_WOMENS)
pd.read_csv(csv_path).head()

### Step 4 — Merge basic + advanced

In [ ]:
from shared.parser import combine_basic_advanced

for key in teams_to_run:
    try:
        combine_basic_advanced(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except FileNotFoundError as e:
        print(f"✗ {key}: {e}")

csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_merged.csv", IS_WOMENS)
merged_df = pd.read_csv(csv_path)
print(f"Merged shape: {merged_df.shape}")
merged_df.head()

### Step 5 — Compute moving averages

In [ ]:
from shared.parser import generate_moving_averages

for span in SPANS:
    for key in teams_to_run:
        try:
            generate_moving_averages(key, SEASON, span, keep_latest=True, is_womens=IS_WOMENS)
            print(f"✓ {key} (span={span})")
        except FileNotFoundError as e:
            print(f"✗ {key} (span={span}): {e}")

# Preview one
csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_5ma.csv", IS_WOMENS)
ma_df = pd.read_csv(csv_path)
print(f"MA shape: {ma_df.shape}")
ma_df.tail(3)

### Step 6 — Run the full team stats pipeline

This runs all steps end-to-end and produces the JSON output.
Without `AZURE_STORAGE_CONNECTION_STRING`, results are written locally.

In [ ]:
from shared.stats import generate_and_upload_team_stats

# Run the full pipeline (writes local JSON if no connection string)
generate_and_upload_team_stats(SEASON)
print("Done — check for ncaam_basketball_team_stats.json / ncaaw_basketball_team_stats.json")

---
## Top 25 Pipeline

Generates AP Top 25 power rankings by running pairwise predictions through the API.

⚠️ Requires the API to be running (either locally or the deployed endpoint).

In [ ]:
from shared.scraper import get_ap_top_25

top25_men = get_ap_top_25(SEASON, is_womens=False)
print(f"Men's AP Top 25: {len(top25_men)} teams")
for i, team in enumerate(top25_men, 1):
    print(f"  #{i} {team}")

In [ ]:
from top25.generate import generate_and_upload_top25

# Use deployed API by default; change to http://localhost:8000 for local
API_URL = "https://mlmb-api.purplesand-9a1718e2.eastus.azurecontainerapps.io"

generate_and_upload_top25(API_URL, SEASON)
print("Done — check for ncaam_basketball_top25.json / ncaaw_basketball_top25.json")

---
## Utilities

Helpers for inspecting data at any point.

In [ ]:
# Inspect any team's files for the current season
def list_team_files(school_key: str, season: int = SEASON, is_womens: bool = IS_WOMENS):
    """List all generated files for a team/season."""
    from shared.scraper import get_data_dir
    data_dir = get_data_dir(school_key, season, is_womens)
    if os.path.exists(data_dir):
        files = os.listdir(data_dir)
        print(f"{data_dir}:")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(data_dir, f))
            print(f"  {f} ({size:,} bytes)")
    else:
        print(f"No data directory for {school_key} ({season})")

list_team_files(ALL_TEAMS[0])

In [ ]:
# Quick preview of any team CSV
def preview_team_csv(school_key: str, suffix: str = "merged", season: int = SEASON, is_womens: bool = IS_WOMENS):
    """Load and display a team CSV. suffix: basic, advanced, merged, 5ma, 5span_full, etc."""
    csv_path = get_team_season_file_path(school_key, season, f"{school_key}_{suffix}.csv", is_womens)
    df = pd.read_csv(csv_path)
    print(f"{csv_path}")
    print(f"Shape: {df.shape}")
    return df

preview_team_csv(ALL_TEAMS[0], "merged")